# Uitleg van de hele code en het dashboard

**Voor iemand die ongeveer 1 maand een data-analyse cursus heeft gevolgd**

In dit notebook leggen we stap voor stap uit:
- welke bestanden belangrijk zijn
- hoe de data wordt opgehaald en samengevoegd
- hoe de Streamlit-app is opgebouwd
- wat elke grafiek en tabel doet
- welke stukken code je als beginner echt moet begrijpen

We houden de uitleg bewust simpel. Het doel is niet om moeilijke woorden te gebruiken,
maar om te snappen **wat de code doet** en **waarom**.

In [2]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, r"c:\Users\AAchr\Documents\Casus 2 phyton minor\ROC")

from data import laad_alle_data, ADVIES_TYPEN, SCHOOLJAREN

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

wijken_df, duo_df, scholen_df, bronnen = laad_alle_data()

print("Bronnen:")
for naam, bron in bronnen.items():
    print(f"- {naam}: {bron}")

print()
print("Aantal rijen:")
print("wijken_df:", len(wijken_df))
print("duo_df:", len(duo_df))
print("scholen_df:", len(scholen_df))

2026-04-06 18:43:02.141 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-06 18:43:02.143 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-06 18:43:02.144 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-06 18:43:02.146 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-06 18:43:02.147 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-06 18:43:02.147 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-06 18:43:02.148 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-06 18:43:02.366 WARNING streamlit.runtime.caching.cache_d

Bronnen:
- CBS Kerncijfers Wijken en Buurten 2024: Live data van CBS OData API (samengevoegd naar dashboardwijken)
- DUO Schooladviezen per school: Live data van DUO Open Onderwijsdata (adviescodes samengevoegd, bijstelling niet apart beschikbaar)
- Amsterdam schoolgebouwen: Live data van Amsterdam schoolgebouwen API

Aantal rijen:
wijken_df: 22
duo_df: 2971
scholen_df: 214


## 1. Welke bestanden zijn belangrijk?

Voor dit project zijn vooral deze 2 bestanden belangrijk:

- `data.py`
  Hier wordt de data opgehaald, opgeschoond en samengevoegd.
- `app.py`
  Hier wordt de Streamlit-app gebouwd: tabs, filters, grafieken en tabellen.

Je kunt het zo zien:

- `data.py` = de **keuken**
- `app.py` = het **restaurant**

Eerst wordt in de keuken alles voorbereid. Daarna laat het restaurant het netjes aan de gebruiker zien.

In [3]:
def toon_regels(bestand, start, einde):
    """Laat een stukje code zien met regelnummers."""
    regels = Path(bestand).read_text(encoding="utf-8").splitlines()
    for i in range(start - 1, einde):
        print(f"{i+1:>4}: {regels[i]}")

## 2. Uitleg van `data.py`

Dit bestand doet 4 hoofdzaken:

1. vaste lijsten en kleuren maken
2. live data ophalen van CBS
3. live data ophalen van DUO en Amsterdam
4. alles samenvoegen tot datasets die de app kan gebruiken

Dat zijn goede beginnerstappen:
- eerst data ophalen
- dan kolommen gelijk maken
- dan datasets koppelen
- dan pas grafieken bouwen

In [ ]:
print("Belangrijke start van data.py")
toon_regels("data.py", 1, 120)

### 2.1 Vaste lijsten en kleuren

Bovenin `data.py` staan vaste dingen zoals:
- `ADVIES_TYPEN`
- `ADVIES_KLEUREN`
- `SCHOOLJAREN`

Waarom is dat handig?

Omdat je die informatie daarna op meerdere plekken opnieuw kunt gebruiken.
Bijvoorbeeld:
- in filters
- in grafieken
- in legenda's
- in berekeningen

Dat voorkomt dat je op 5 plekken steeds dezelfde lijst opnieuw moet typen.

### 2.2 CBS-data

De functie `haal_cbs_data_op()` haalt wijkgegevens op, zoals:
- inkomen
- aantal inwoners
- aandeel niet-westerse achtergrond
- opleidingsniveau
- uitkering
- WOZ-waarde

Belangrijk om te begrijpen:

De live CBS-data gebruikt **andere wijkcodes** dan jullie dashboard.
Daarom wordt de live data later omgezet naar jullie eigen wijkgroepen.
Dat is een heel normale stap in data-analyse:
de brondata past niet altijd meteen op jouw onderzoeksvraag.

In [ ]:
print("CBS-functies in data.py")
toon_regels("data.py", 40, 185)

### 2.3 DUO-data

De DUO-data gaat over schooladviezen.

Daar zie je per school en per jaar:
- hoeveel leerlingen een bepaald advies kregen

In jullie code worden numerieke DUO-codes vertaald naar begrijpelijke adviesgroepen, zoals:
- `VMBO-BBL`
- `VMBO-KBL`
- `VMBO-TL`
- `HAVO`
- `HAVO/VWO`
- `VWO`

Dat is belangrijk, want gebruikers willen geen code `7` of `11` zien,
maar een echte naam.

In [1]:
print("DUO-functies in data.py")
toon_regels("data.py", 150, 290)

DUO-functies in data.py


NameError: name 'toon_regels' is not defined

### 2.4 Amsterdam schooldata

De Amsterdam API levert schoolinformatie zoals:
- schoolnaam
- BRIN
- stadsdeel
- wijk

Die data is vooral handig om scholen op de kaart te zetten en om DUO-data aan Amsterdamse scholen te koppelen.

Let op: in deze app worden schoollocaties **benaderd** met wijk- of stadsdeelcentra.
Dat betekent:
- de school staat wel ongeveer goed op de kaart
- maar niet exact op het echte gebouw

In [ ]:
print("Amsterdam schooldata en koppeling")
toon_regels("data.py", 290, 430)

### 2.5 Alles samenvoegen in `laad_alle_data()`

Dit is de belangrijkste functie van `data.py`.

Daar gebeurt ongeveer dit:

1. haal CBS, DUO en Amsterdam data op
2. koppel scholen aan wijken
3. bereken percentages
4. maak een samenvatting per wijk
5. geef de datasets terug aan de app

Het resultaat zijn 3 tabellen:

- `wijken_df`
  Alles op wijkniveau
- `duo_df`
  Alles op school x jaar x adviestype
- `scholen_df`
  Schoolnamen en kaartlocaties

In [ ]:
print("Belangrijkste samenvoegfunctie")
toon_regels("data.py", 430, 520)

print("\nVoorbeeld van wijken_df")
display(wijken_df[["wijk_naam", "stadsdeel", "gem_inkomen", "pct_hoog_advies", "pct_laag_advies"]].head())

print("Voorbeeld van duo_df")
display(duo_df[["school_naam", "wijk_naam", "schooljaar", "advies_type", "aantal_leerlingen", "pct"]].head())

print("Voorbeeld van scholen_df")
display(scholen_df.head())

## 3. Uitleg van `app.py`

`app.py` is de presentatie-laag.

Daar gebeurt dit:

1. pagina-instellingen zetten
2. data inladen
3. filters in de sidebar maken
4. tabs maken
5. grafieken en tabellen tonen

Dit is precies hoe veel dashboards zijn opgebouwd.

In [ ]:
print("Start van app.py")
toon_regels("app.py", 1, 120)

### 3.1 De sidebar met filters

In de sidebar kiest de gebruiker:
- stadsdeel
- wijk
- schooljaren
- adviestypen

Daarna worden de datasets gefilterd.

Dit is een heel belangrijk idee in dashboards:

- **eerst filteren**
- **daarna pas tekenen**

Zo veranderen alle grafieken automatisch mee.

In [ ]:
print("Filters en tabdefinitie")
toon_regels("app.py", 120, 210)

## 4. Uitleg per tab, grafiek en tabel

Hieronder leggen we per onderdeel uit:
- wat je ziet
- welke data gebruikt wordt
- hoe je de grafiek moet lezen

### Tab 1 - Introductie

Deze tab bevat vooral tekst en 4 kerncijfers:
- gemiddelde % hoog advies
- hoogste wijk
- laagste wijk
- verschil tussen wijken

Waarom is dit handig?

Omdat een gebruiker meteen het grote verhaal ziet voordat hij naar detailgrafieken gaat kijken.

In [ ]:
toon_regels("app.py", 210, 275)

### Tab 2 - Kaart Amsterdam

Hier staan 2 kaartdelen:

1. **Wijkkaart**
   Elke wijk krijgt een kleur op basis van `% hoog advies`.

2. **Schoollocaties**
   Scholen worden als punten op de kaart gezet.

Hoe lees je dit?

- groener = meer hoge adviezen
- roder = minder hoge adviezen
- grotere punten = hoger percentage hoog advies

Dit is een visuele manier om verschillen tussen wijken snel te zien.

In [ ]:
toon_regels("app.py", 275, 345)

### Tab 3 - Schooladviezen

Deze tab heeft 3 onderdelen:

1. **Gestapelde staafgrafiek**
   Laat per wijk zien hoe de adviesverdeling is opgebouwd.

2. **Scatterplot inkomen vs hoog advies**
   Laat zien of rijkere wijken ook meer hoge adviezen hebben.

3. **Heatmap**
   Laat per wijk en per advies_type het percentage zien.

Waarom is dit nuttig?

Omdat je hetzelfde onderwerp op 3 manieren bekijkt:
- verdeling
- verband
- overzicht

In [ ]:
toon_regels("app.py", 345, 455)

### Tab 4 - Doorstroomtoets

Deze tab vergelijkt:
- `2022-2023` = oude situatie
- `2023-2024` = eerste jaar doorstroomtoets

Grafieken:

1. **Voor/na staafgrafiek**
   Vergelijkt adviesverdelingen tussen de 2 jaren.

2. **Verschilgrafiek**
   Laat direct zien welke adviezen stegen of daalden.

3. **Bijgesteld advies per wijk**
   Laat zien in welke wijken het meeste omhoog is bijgesteld.

Als beginner moet je hier vooral snappen:
- je vergelijkt twee jaren
- je berekent verschillen
- je laat die verschillen visueel zien

In [ ]:
toon_regels("app.py", 455, 560)

### Tab 5 - Voorspelling

Hier wordt een simpel regressiemodel gebruikt.

Het model probeert `% hoog advies` te voorspellen op basis van:
- `gem_inkomen`
- `pct_niet_westers`
- `pct_uitkering`
- `pct_laag_opgeleid`

Belangrijke woorden:

- **voorspeld** = wat het model denkt
- **werkelijk** = wat echt in de data staat
- **residu** = verschil tussen die twee

Waarom is dat nuttig?

Omdat je zo kunt zien welke wijken beter of slechter scoren dan je op basis van hun achtergrond zou verwachten.

In [ ]:
toon_regels("app.py", 560, 705)

### Tab 6 - Data & Methoden

Dit is de uitleg-tab van het dashboard zelf.

Hier laat de app zien:
- welke databronnen zijn gebruikt
- hoe de APIs werken
- hoe datasets zijn gekoppeld
- ruwe tabellen

Waarom is dit belangrijk?

Omdat een dashboard niet alleen mooie grafieken moet tonen,
maar ook eerlijk moet laten zien **waar de cijfers vandaan komen**.

In [ ]:
toon_regels("app.py", 705, 860)

## 5. Uitleg van de tabellen in de app

Onderaan de app staan ook tabellen met ruwe data.

Dat is goed, want:
- grafieken geven een samenvatting
- tabellen laten de echte waarden zien

Voor beginners is dit een mooie regel:

**Gebruik een grafiek om een verhaal te vertellen, en een tabel om te controleren of het verhaal klopt.**

In [ ]:
print("Voorbeeld van een wijktabel")
display(wijken_df[["wijk_naam", "stadsdeel", "gem_inkomen", "pct_hoog_advies", "pct_laag_advies"]].head(10))

print("Voorbeeld van een schooladviestabel")
display(duo_df[["school_naam", "wijk_naam", "schooljaar", "advies_type", "aantal_leerlingen", "pct"]].head(10))

## 6. Wat moet je als beginner vooral onthouden?

Als je pas kort met data-analyse bezig bent, probeer dan vooral dit te onthouden:

1. Data ophalen is vaak maar het begin.
   Je moet data daarna bijna altijd nog opschonen en koppelen.

2. Een dashboard bestaat uit kleine stappen.
   Eerst data inladen, dan filteren, dan berekenen, dan tekenen.

3. Een grafiek is een vraag in beeldvorm.
   Vraag jezelf altijd af: wat wil ik hier laten zien?

4. Niet alles hoeft meteen perfect of slim.
   Duidelijke, simpele code is vaak beter dan ingewikkelde code.

5. Tabellen blijven belangrijk.
   Daarmee kun je controleren of je grafiek logisch is.

Als je deze 5 dingen begrijpt, dan begrijp je de basis van dit project al goed.